In [1]:

import rasterio
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
import openeo

Conectarse al API usando los datos de usuario y contraseña

In [3]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=JYOV-RDNL 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.


In [4]:
#Areas de interes
lago_atitlan = {
    "west": -91.326256,
    "east": -91.07151,
    "south": 14.5948,
    "north": 14.750979
}
lago_amatitlan = {
    "west": -90.638065,
    "east": -90.512924, 
    "south": 14.412347,
    "north": 14.493799
}


## Ejercicio 2: Obtención de Datos Raster

Usando `openEO` para descargar B03, B04, B05 y B08 de la colección `SENTINEL2_L2A` para las fechas especificadas.

In [5]:
import os
import datetime
from datetime import date

os.makedirs("data", exist_ok=True)

fechas_atitlan = [
    "2025-01-18", "2025-04-13", "2025-05-13", "2025-07-17",
    "2025-11-21", "2025-12-29", "2026-02-12", "2026-03-24",
    "2026-04-13", "2026-04-28", "2026-07-22"
]

fechas_amatitlan = [
    "2025-01-28", "2025-04-15", "2025-04-28", "2025-11-24",
    "2026-01-08", "2026-02-02", "2026-02-07", "2026-03-29",
    "2026-04-13", "2026-04-28", "2026-06-19"
]

lagos = {
    "atitlan": {"bbox": lago_atitlan, "fechas": fechas_atitlan},
    "amatitlan": {"bbox": lago_amatitlan, "fechas": fechas_amatitlan},
}

def descargar_bandas(lago, bbox, fecha):
    """Descarga B03, B04, B05, B08 para un lago y una fecha, si no existe ya localmente."""
    out_path = f"data/{lago}_{fecha}.tif"
    if os.path.exists(out_path):
        return out_path

    d0 = date.fromisoformat(fecha)
    d1 = d0 + datetime.timedelta(days=1)  # ventana de 1 dia para que openeo encuentre la escena

    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=[str(d0), str(d1)],
        bands=["B03", "B04", "B05", "B08"],
    )
    cube = cube.reduce_dimension(dimension="t", reducer="mean")
    cube.download(out_path)
    return out_path


In [ ]:
# Descarga para ambos lagos (puede tardar varios minutos)
for lago, info in lagos.items():
    for fecha in info["fechas"]:
        try:
            path = descargar_bandas(lago, info["bbox"], fecha)
            print(f"OK: {path}")
        except Exception as e:
            print(f"FALLO {lago} {fecha}: {e}")


OK: data/atitlan_2025-01-18.tif
